In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import os
from PIL import Image
import keras
from keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.layers import Conv2D, MaxPooling2D, Input, Conv2DTranspose, Concatenate, BatchNormalization, UpSampling2D
from keras.layers import  Dropout, Activation, SpatialDropout2D
from keras.layers import concatenate
from keras.optimizers import Adam, SGD, RMSprop
from keras.layers import ELU, PReLU, LeakyReLU
from keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from keras import backend as K
from keras.utils import plot_model
import tensorflow as tf
import glob
import random
import cv2
from random import shuffle
import statistics

import types
# --- remendo Keras 2.10: cria keras.saving.register_keras_serializable se faltar ---
if not hasattr(keras, "saving"):
    keras.saving = types.SimpleNamespace()
if not hasattr(keras.saving, "register_keras_serializable"):
    keras.saving.register_keras_serializable = tf.keras.utils.register_keras_serializable


In [3]:
# VARIAVEIS GERAIS
batch_size = 16
img_sz_x = 256
img_sz_y = 256

# --- identifica qual par de modelos comparar (offline vs online) ---
dataset      = "crop_OralEpitheliumDB_aug"   # seu dataset
loss_online  = "combo_loss"                  # a loss treinada COM ProtoSeg
loss_offline = "bce"                         # o sem-ProtoSeg foi salvo como bce
seed         = 42


In [4]:
class ProtoSeg(nn.Module):
    def __init__(self,ndims='2d'):
        super().__init__()

        # for 1D: self.dims=(2)
        # for 2D image: self.dims=(2,3)
        # for 3D image: self.dims=(2,3,4)
        if ndims == '1d':
            self.dims = (2)
        elif ndims == '2d':
            self.dims = (2,3)
        elif ndims == '3d':
            self.dims = (2,3,4)
        else:
            raise ValueError('ndims must be in [1d,2d,3d]')

        self.softmax = nn.Softmax(dim=1)

    def forward(self,xfeat,pred,mask=None):
        #@ xfeat: the deep feature need to be inperpreted
        #@ pred: the initial segmentation results from the last layer of the network
        #@ mask is to maks out the background of the image (without any tissue)

        if mask is not None:
            pos_prototype = torch.sum(xfeat*pred*mask,dim=self.dims,keepdim=True)
            num_prototype = torch.sum(pred*mask,dim=self.dims,keepdim=True)
            pos_prototype = pos_prototype / num_prototype

            rpred = 1 - pred

            neg_prototype = torch.sum(xfeat*rpred*mask,dim=self.dims,keepdim=True)
            num_prototype = torch.sum(rpred*mask,dim=self.dims,keepdim=True)
            neg_prototype = neg_prototype / num_prototype

            pfeat = -torch.pow(xfeat-pos_prototype,2).sum(1,keepdim=True)
            nfeat = -torch.pow(xfeat-neg_prototype,2).sum(1,keepdim=True)

            disfeat = torch.cat([nfeat,pfeat],dim=1)
            pred = self.softmax(disfeat)

        else:
            pos_prototype = torch.sum(xfeat*pred,dim=self.dims,keepdim=True)
            num_prototype = torch.sum(pred,dim=self.dims,keepdim=True)
            pos_prototype = pos_prototype / num_prototype

            rpred = 1 - pred

            neg_prototype = torch.sum(xfeat*rpred,dim=self.dims,keepdim=True)
            num_prototype = torch.sum(rpred,dim=self.dims,keepdim=True)
            neg_prototype = neg_prototype / num_prototype

            pfeat = -torch.pow(xfeat-pos_prototype,2).sum(1,keepdim=True)
            nfeat = -torch.pow(xfeat-neg_prototype,2).sum(1,keepdim=True)

            disfeat = torch.cat([nfeat,pfeat],dim=1)
            pred = self.softmax(disfeat)

        return pred

In [5]:
def dice_score(mask_true, mask_pred):
    mask_true = tf.cast(mask_true, tf.float32)
    mask_pred = tf.cast(mask_pred, tf.float32)
    intersection = tf.reduce_sum(mask_true * mask_pred)
    dice = (2.0 * intersection) / (tf.reduce_sum(mask_true) + tf.reduce_sum(mask_pred))
    return dice

@keras.saving.register_keras_serializable()
def SA_score(y_true, y_pred): # images, masks, outputs, feature_extractor

    img = current_X
    mask = y_true
    output = current_Y

    feature_extractor = Model(
       inputs=model.inputs,
       outputs=model.layers[-1].output,
    )

    x = tf.reshape(img, (1, 256, 256, 3))
    features = feature_extractor(x)
    x = tf.transpose(features, (0, 3, 1, 2))

    pred_map = tf.transpose(output, (2, 0, 1))
    pred_map = tf.reshape(pred_map, (1, 1, 256, 256))

    shape = int(x.shape[2])

    if shape != 256:
        x = tf.image.resize(x, size=(256, 256), method='bilinear')

    neters = ProtoSeg(ndims='2d')
    probability_map = neters(x, pred_map, mask=None)
    binary_map = tf.argmax(probability_map, 1)

    bmap = tf.transpose(binary_map, (1, 2, 0))

    sa_score = dice_score(mask, bmap)

    return sa_score

In [6]:
@keras.saving.register_keras_serializable()
def mean_iou(y_true, y_pred):
    yt0 = y_true[:,:,:,0]
    yp0 = K.cast(y_pred[:,:,:,0] > 0.5, 'float32')
    inter = tf.math.count_nonzero(tf.logical_and(tf.equal(yt0, 1), tf.equal(yp0, 1)))
    union = tf.math.count_nonzero(tf.add(yt0, yp0))
    iou = tf.where(tf.equal(union, 0), 1., tf.cast(inter/union, 'float32'))
    return iou

@keras.saving.register_keras_serializable()
def custom_loss(y_true, y_pred):
    # Exemplo simples: entropia cruzada binária
    sa_score = SA_score(y_true, y_pred)

    if K.any(tf.math.is_nan(sa_score)):
        sa_score = tf.constant(0, dtype=tf.float32)
        #sa_score = tf.constant(1e10)
    loss = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return tf.add(loss, sa_score)

In [7]:
# ===== Arquitetura U-Net (mesma do treino) — necessária para load_weights =====
def unet(sz = (img_sz_x, img_sz_y, 3), learning_rate=0.001):
  inputs = Input(sz)
  l2_reg = 0.001
  conv1 = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
  conv1 = Conv2D(32, (3, 3), activation='relu', padding='same')(conv1)
  pool1 = MaxPooling2D((2, 2))(conv1)
  conv2 = Conv2D(64, (3, 3), activation='relu', padding='same')(pool1)
  conv2 = Conv2D(64, (3, 3), activation='relu', padding='same')(conv2)
  drop2 = SpatialDropout2D(0.50)(conv2)
  pool2 = MaxPooling2D((2, 2))(drop2)
  conv3 = Conv2D(128, (3, 3), activation='relu', padding='same')(pool2)
  conv3 = Conv2D(128, (3, 3), activation='relu', padding='same')(conv3)
  drop3 = SpatialDropout2D(0.50)(conv3)
  pool3 = MaxPooling2D((2, 2))(drop3)
  conv4 = Conv2D(256, (3, 3), activation='relu', padding='same')(pool3)
  conv4 = Conv2D(256, (3, 3), activation='relu', padding='same')(conv4)
  drop4 = SpatialDropout2D(0.50)(conv4)
  pool4 = MaxPooling2D((2, 2))(drop4)
  conv5 = Conv2D(512, (3, 3), activation='relu', padding='same')(pool4)
  conv5 = Conv2D(512, (3, 3), activation='relu', padding='same')(conv5)
  drop5 = SpatialDropout2D(0.50)(conv5)
  up6 = Conv2DTranspose(512, (2, 2), strides=(2, 2), padding='same')(drop5)
  merge6 = concatenate([up6, drop4], axis=3)
  conv6 = Conv2D(256, (3, 3), activation='relu', padding='same')(merge6)
  conv6 = Conv2D(256, (3, 3), activation='relu', padding='same')(conv6)
  up7 = Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(conv6)
  merge7 = concatenate([up7, drop3], axis=3)
  conv7 = Conv2D(128, (3, 3), activation='relu', padding='same')(merge7)
  conv7 = Conv2D(128, (3, 3), activation='relu', padding='same')(conv7)
  up8 = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(conv7)
  merge8 = concatenate([up8, drop2], axis=3)
  conv8 = Conv2D(64, (3, 3), activation='relu', padding='same')(merge8)
  conv8 = Conv2D(64, (3, 3), activation='relu', padding='same')(conv8)
  up9 = Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(conv8)
  merge9 = concatenate([up9], axis=3)
  conv9 = Conv2D(32, (3, 3), activation='relu', padding='same')(merge9)
  conv9 = Conv2D(32, (3, 3), activation='relu', padding='same')(conv9)
  output = Conv2D(1, (1, 1), activation='sigmoid')(conv9)
  model = Model(inputs, output)
  return model


In [8]:
from random import shuffle
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
from PIL import Image
import cv2

current_X = {}
current_Y = {}

def image_generator(group, files, batch_size = 4, sz = (256, 256), mendeley=False):
    global current_X, current_Y

    while True:
        #extract a random batch
        batch = np.random.choice(files, size = batch_size)

        #variables for collecting batches of inputs and outputs
        batch_x = []
        batch_y = []

        for f in batch:

            #get the masks. Note that masks are png files
            mask = Image.open(f'datasets/{dataset}/{group}/mascaras/{f}.png')

            mask = np.array(mask.resize(sz))
            mask = np.expand_dims(mask, axis=-1)

            #preprocess the mask
            mask[mask > 0] = 1
            batch_y.append(mask)

            # Carregar a imagem
            raw = cv2.imread(f'datasets/{dataset}/{group}/{f}.png')
            raw = cv2.resize(raw, sz)
            raw = np.array(raw)

            #check the number of channels because some of the images are RGBA or GRAY
            if len(raw.shape) == 2:
              raw = np.stack((raw,)*3, axis=-1)

            raw = raw[:, :, 0:3]

            batch_x.append(raw)

        #preprocess a batch of images and masks
        batch_x = np.array(batch_x)/255.
        batch_y = np.array(batch_y)
        #batch_y = np.expand_dims(batch_y,3)

        #current_X = batch_x[0]
        #current_Y = batch_y[0]

        current_X = batch_x
        current_Y = batch_y

        yield (batch_x, batch_y)

In [9]:
import os

all_files = []

test_files = list(os.path.splitext(file)[0] for file in os.listdir(f'datasets/{dataset}/test') if os.path.isfile(os.path.join(f'datasets/{dataset}/test', file)))

test_generator  = image_generator("test", test_files, batch_size = batch_size)

In [10]:
def dice_score(mask_true, mask_pred):
    intersection = np.sum(mask_true * mask_pred)
    dice = (2.0 * intersection) / (np.sum(mask_true) + np.sum(mask_pred))
    return dice

In [11]:
def get_SA_scores(images, masks, outputs, feature_extractor):
    '''
    images: imagens originais
    masks: mascaras originais
    outputs: mascaras geradas pelo modelo
    '''

    sa_scores = []

    for i in range(0, len(images)):

        img = images[i]

        mask = masks[i]
        output = outputs[i]

        x = img.reshape((1, 256, 256, 3))
        features = feature_extractor(x)
        x = np.transpose(features, (0, 3, 1, 2))
        x = torch.from_numpy(x)

        pred_map = np.transpose(output, (2, 0, 1))
        pred_map = pred_map.reshape((1, 1, 256, 256))
        pred_map = torch.from_numpy(pred_map)

        shape = int(x.shape[2])

        if shape != 256:
            x = F.interpolate(x, size=(256, 256), mode='bilinear', align_corners=False)

        neters = ProtoSeg(ndims='2d')
        probability_map = neters(x, pred_map, mask=None)
        binary_map = torch.argmax(probability_map, 1)

        bmap = np.transpose(np.asarray(binary_map), (1, 2, 0))

        sa_score = dice_score(mask, bmap)
        sa_scores.append(sa_score)

    return sa_scores

In [12]:
import os

custom_objects = {"mean_iou": mean_iou}

dataset_id = dataset.replace('/', '_')
run_offline = f"Unet__{dataset_id}__proto-offline__loss-{loss_offline}__seed{seed}"
run_online  = f"Unet__{dataset_id}__proto-online__loss-{loss_online}__seed{seed}"

# caminho dos PESOS dentro da pasta de cada rodada (novo padrão de salvamento)
modelos = [
    os.path.join("saida", "runs", run_offline, "pesos.weights.h5"),
    os.path.join("saida", "runs", run_online,  "pesos.weights.h5"),
]

for m in modelos:
    print(("OK   " if os.path.exists(m) else "FALTA"), m)

camadas_modelos = []
medias_modelos = []
desvios_modelos = []


OK    saida\runs\Unet__crop_OralEpitheliumDB_aug__proto-offline__loss-bce__seed42\pesos.weights.h5
OK    saida\runs\Unet__crop_OralEpitheliumDB_aug__proto-online__loss-combo_loss__seed42\pesos.weights.h5


In [13]:
for j in range(0, 10):

  for modelo in modelos:
      # reconstruir a arquitetura e carregar os PESOS (não é modelo completo)
      model = unet(sz=(img_sz_x, img_sz_y, 3))
      model.load_weights(modelo)

      images, masks = next(test_generator)

      masks_teste = model.predict(images)
      print("Shape Classes:", masks_teste.shape)

      camadas = []
      medias  = []
      desvios = []

      for i in range(0, len(model.layers)):
          layer = str(model.layers[i])
          if "Conv2D" in layer and not "transpose" in layer:
              if "relu" in str(model.layers[i].activation) or "sigmoid" in str(model.layers[i].activation):
                  feature_extractor = Model(
                    inputs=model.inputs,
                    outputs=model.layers[i].output,
                  )

                  scores = get_SA_scores(images, masks, masks_teste, feature_extractor)

                  if len(scores) > 0:
                      media = statistics.mean(scores)
                      desvio_padrao = statistics.stdev(scores)

                      camadas.append(i)
                      medias.append(media)
                      desvios.append(desvio_padrao)

      if j == 0:
        camadas_modelos.append(camadas)
      medias_modelos.append(medias)
      desvios_modelos.append(desvios)


1/1 [==============================] - 13s 13s/step
Shape Classes: (16, 256, 256, 1)


C:\Users\Dell\AppData\Local\Temp\ipykernel_8148\2984796420.py:20: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  x = torch.from_numpy(x)


1/1 [==============================] - 0s 471ms/step
Shape Classes: (16, 256, 256, 1)
1/1 [==============================] - 0s 444ms/step
Shape Classes: (16, 256, 256, 1)
1/1 [==============================] - 0s 328ms/step
Shape Classes: (16, 256, 256, 1)
1/1 [==============================] - 0s 324ms/step
Shape Classes: (16, 256, 256, 1)
1/1 [==============================] - 0s 355ms/step
Shape Classes: (16, 256, 256, 1)
1/1 [==============================] - 0s 312ms/step
Shape Classes: (16, 256, 256, 1)
1/1 [==============================] - 0s 317ms/step
Shape Classes: (16, 256, 256, 1)
1/1 [==============================] - 0s 302ms/step
Shape Classes: (16, 256, 256, 1)
1/1 [==============================] - 0s 280ms/step
Shape Classes: (16, 256, 256, 1)
1/1 [==============================] - 0s 287ms/step
Shape Classes: (16, 256, 256, 1)
1/1 [==============================] - 0s 286ms/step
Shape Classes: (16, 256, 256, 1)
1/1 [==============================] - 0s 309ms/step
S

In [1]:
def calcula_media(lista):

  # Separar listas de índices pares e ímpares
  pares = lista[::2]
  impares = lista[1::2]

  # Calcular médias
  media_pares = np.mean(pares, axis=0)
  media_impares = np.mean(impares, axis=0)

  return [media_pares.tolist(), media_impares.tolist()]

medias_modelos = calcula_media(medias_modelos)
desvios_modelos = calcula_media(desvios_modelos)

NameError: name 'medias_modelos' is not defined

In [ ]:
print(len(camadas_modelos[0]))

camadas = list(range(1, len(camadas_modelos[0]) + 1))

print(camadas)

In [ ]:
icones = ['-^', '-s']
ecores = ['#b2b2d9', '#b2d9b2']
cores = ['#000080', '#008000']
labels = ['ProtoSeg Offline', 'ProtoSeg Online']

line_style = {'alpha': 0.5}
icon_style = {}

fig, ax = plt.subplots(figsize=(7, 4))

for i in range(len(medias_modelos)):
    plt.errorbar(list(map(lambda x: x + (0.25*i), camadas)),
                 medias_modelos[i], yerr=desvios_modelos[i],
                 fmt=icones[i], capsize=2, elinewidth=1, linewidth=2, ecolor=ecores[i], color=cores[i], label=labels[i])

plt.xlabel('Camada Convolucional')
plt.ylabel('SA Score')
#plt.title('Gráfico com Desvios Padrão')
plt.legend(loc="lower right")

plt.xticks([1, 5, 10, 15, len(camadas_modelos[0])])


os.makedirs("saida/imagens", exist_ok=True)
nome_fig = f"Camadas_{dataset_id}__{loss_online}__seed{seed}"
plt.savefig(f"saida/imagens/{nome_fig}.svg")
plt.savefig(f"saida/imagens/{nome_fig}.png", dpi=150)

plt.show()
print(modelo)